## RAG 3일차

### InsureLLM을 위한 전문 질문-답변 시스템

LangChain 1.0으로 구현한 RAG 파이프라인.

어제 만든 VectorStore를 활용합니다 (HuggingFace `all-MiniLM-L6-v2` 임베딩 사용)

> 💡 **전문가 조언**: LangChain의 진가는 **조합 가능성**에 있습니다. `retriever` → `llm` → `chain` 으로 이어지는 파이프라인을 몇 줄로 구성할 수 있으며, 각 컴포넌트를 독립적으로 교체/테스트할 수 있습니다.

In [1]:
# 필요한 라이브러리 임포트
# langchain_core.messages: LangChain 방식의 메시지 객체 (SystemMessage, HumanMessage)
# Chroma: 어제 생성한 벡터 저장소에 연결하기 위한 모듈
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Chroma에 연결; HuggingFace all-MiniLM-L6-v2 임베딩 사용

> 💡 **전문가 조언**: 임베딩 모델은 반드시 **인제스트 시(저장)와 쿼리 시(검색) 동일한 모델을 사용**해야 합니다. 다른 모델을 사용하면 벡터 공간이 달라져 검색 결과가 완전히 엉망이 됩니다.

In [3]:
# day2에서 생성한 벡터 저장소(vector_db)를 로드합니다
# 인제스트 때와 동일한 임베딩 모델을 반드시 사용해야 합니다
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### LangChain의 핵심 객체 2가지 설정: retriever와 llm

#### "temperature(온도)"에 대한 간단 설명:
- 출력의 다양성을 조절합니다
- temperature=0은 출력이 예측 가능(결정론적)하다는 의미
- 높은 temperature일수록 답변에 다양성 증가

temperature를 '창의성'으로 설명하는 사람도 있지만 정확한 표현은 아닙니다
- 실제로는 추론 시 어떤 토큰이 선택되는지를 제어합니다
- temperature=0: 항상 가장 높은 확률의 토큰 선택
- temperature=1: 10% 확률의 토큰이 실제로 10% 빈도로 선택됨

참고: temperature=0이어도 출력이 항상 재현 가능한 것은 아닙니다. 재현성을 보장하려면 random seed도 설정해야 합니다. 6~8주차에서 다룰 예정입니다. (그럼에도 항상 재현 가능하지는 않습니다.)

참고 2: 창의성을 원한다면, System Prompt를 활용하세요!

> 💡 **전문가 조언**: RAG 기반 Q&A 시스템에서는 `temperature=0`이 일반적으로 권장됩니다. 사실에 기반한 정확한 답변이 필요하기 때문입니다. 반면 창작이나 브레인스토밍 도구라면 높은 temperature가 오히려 유리합니다.

In [4]:
# retriever: 벡터 유사도 검색으로 관련 문서를 가져오는 객체
# llm: temperature=0으로 설정하여 사실 기반의 일관된 답변 생성
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### 이 LangChain 객체들은 모두 `invoke()` 메서드를 구현합니다

> 💡 **전문가 조언**: LangChain의 모든 주요 컴포넌트(retriever, llm, chain 등)는 동일한 `invoke()` 인터페이스를 따릅니다. 이를 통해 컴포넌트를 파이프라인(`|` 연산자)으로 쉽게 연결할 수 있습니다.

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='9fb27c10-ec12-4719-a8d9-6b2d8c74389b', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content="Avery is a given name that can be used for both males and females. It can also be a surname. Without additional context, it's difficult to determine which specific Avery you're referring to. If you can provide more details—such as a full name, profession, or the context in which Avery is mentioned—I’d be happy to help you with more specific information.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 11, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_5ee6113273', 'id': 'chatcmpl-DQNQ8fjijz0rFBTCajXOi5v8uuUI8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d50e8-cef1-70f2-8d74-64c450d7de38-0',

## 이제 모두 합쳐봅시다!

> 💡 **전문가 조언**: RAG의 핵심 흐름 — ① 질문 입력 → ② 관련 문서 검색(retriever) → ③ 문서를 컨텍스트로 삽입 → ④ LLM 응답 생성. 이 4단계가 RAG의 전부입니다.

In [7]:
# 시스템 프롬프트 템플릿: {context}에 검색된 관련 문서가 동적으로 삽입됨
# 💡 "모르면 모른다고 말하라"는 지시가 환각(hallucination) 억제의 핵심입니다
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
# 전체 RAG 파이프라인을 실행하는 함수
# 1단계: retriever로 관련 문서 검색
# 2단계: 검색된 문서들을 하나의 컨텍스트 문자열로 합침
# 3단계: 시스템 프롬프트에 컨텍스트 삽입 후 LLM 호출
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

'It seems there might be a typo in the name. Based on the information I have, you might be referring to Avery Lancaster. She is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, based in San Francisco, California. Avery has been with the company since 2015 and has played a key role in guiding Insurellm to become a leading Insurance Tech provider. If you meant someone else or need more details, please let me know!'

## 다음에는 무엇이 올까요? 😂

In [10]:
gr.ChatInterface(answer_question).launch()

C:\Users\yeop\anaconda3\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 인정하세요 - RAG가 이것보다 훨씬 복잡할 거라고 생각했죠!!

> 💡 **전문가 정리**: LangChain으로 구현한 기본 RAG는 단 4줄(retriever 생성, 문서 검색, 프롬프트 조립, LLM 호출)로 완성됩니다. 하지만 프로덕션 품질을 위해선 청크 전략, 리랭킹, 쿼리 재작성, 평가(eval) 등 추가 작업이 필요합니다 — 다음 시간에 다룰 내용입니다!